# EU Taxonomy RAG – Interactive Demo & Learning Notebook

This notebook walks through the RAG pipeline built for the EU Taxonomy FAQ.

**What you'll see:**
1. How the raw markdown FAQ is parsed into structured chunks.
2. How the vector store is built with OpenAI embeddings.
3. The retrieval strategy: **classic** dense similarity search over embeddings.
4. How prompt engineering changes the generated answer.
5. Evaluation on the golden QA set with ragas metrics.
6. A playground to test your own questions.
7. The hybrid retrieval pipeline (BM25 + Dense + RRF + Cross-Encoder) and comparison.

**Prerequisites:**
- Install dependencies: `pip install -r ../apps/api/requirements.txt`
- Activate the API virtual environment: `source apps/api/.venv/bin/activate`
- `OPENROUTER_API_KEY` must be set in your environment or in `apps/api/.env`


In [12]:
import os
import sys
from pathlib import Path

# Make the backend importable from the notebook folder
sys.path.insert(0, str(Path.cwd().parent / "apps" / "api"))

# Load API key from .env if present
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / "apps" / "api" / ".env")

os.getenv("OPENROUTER_API_KEY")[:12] + "..." if os.getenv("OPENROUTER_API_KEY") else "⚠️  MISSING – set OPENROUTER_API_KEY"


'sk-or-v1-95d...'

---
## 1. FAQ Parsing

The raw FAQ is a single markdown file. We parse it into a list of `FAQChunk` objects.
Each chunk carries `section`, `question`, `answer`, and an `id`.


In [13]:
from rag.index.methods.basic import parse_taxonomy_faqs, FAQ_PATH, load_documents

chunks = parse_taxonomy_faqs(FAQ_PATH)
docs = load_documents()

print(f"FAQ file: {FAQ_PATH}")
print(f"Total sections: {len({c.section for c in chunks})}")
print(f"Total Q&A chunks: {len(chunks)}")
print()

# Show a sample chunk
sample = chunks[5]
print("Sample chunk:")
print(f"  Section : {sample.section}")
print(f"  Question: {sample.question[:120]}...")
print(f"  Answer  : {sample.answer[:200]}...")
print(f"  ID      : {sample.id}")


FAQ file: /Users/bm/Documents/Projet/RAG EU Taxonomy/apps/api/data/taxonomy_faqs_cleaned.md
Total sections: 7
Total Q&A chunks: 329

Sample chunk:
  Section : Climate Delegated Act
  Question: How should GHG emissions for technical screening criteria be calculated (scope, methodologies etc.)?...
  Answer  : There is no universal calculation method for GHG emissions in the Climate Delegated Act. Specific calculation methods are set out in the TSC for specific activities, in line with the methodological ap...
  ID      : 5


### Section breakdown


In [14]:
from collections import Counter

section_counts = Counter(c.section for c in chunks)
for section, count in section_counts.most_common():
    print(f"{count:3d}  {section}")


195  Climate Delegated Act
 37  Taxonomy-Alignment Reporting
 33  Taxonomy-Eligibility reporting (part 2)
 23  EU Taxonomy - General
 22  Taxonomy-Eligibility reporting (part 1)
 11  Disclosures Delegated Act - General
  8  Complementary Climate Delegated Act


---
## 2. Vector Store & Embeddings

The `classic` retriever relies on `text-embedding-3-small` embeddings stored in ChromaDB.
The first call to `get_vector_store()` indexes all documents if the DB does not yet exist.

> ⚠️ This cell may take 30–60 s the first time because every chunk is embedded via the OpenRouter API.


In [15]:
from rag.index.store import get_vector_store

store = get_vector_store()
print("✅ Vector store ready")
print(f"Collection: {store._collection.name}")
print(f"Documents : {store._collection.count()}")


✅ Vector store ready
Collection: taxonomy_faqs
Documents : 329


---
## 3. Retrieval – Classic Dense Retrieval

We retrieve the top-4 most relevant chunks using dense embedding similarity over ChromaDB.


In [16]:
from rag.query.methods.classic import get_retriever

QUESTION = "Are the technical screening criteria under the Climate Delegated Act going to be tightened and reviewed periodically?"

classic_docs = get_retriever(k=4).invoke(QUESTION)

def print_docs(docs, title):
    print(f"\n{'='*60}")
    print(title)
    print('='*60)
    for i, d in enumerate(docs, 1):
        print(f"\n--- Result {i} ---")
        print(f"Section : {d.metadata['section']}")
        print(f"Question: {d.metadata['question'][:120]}")
        print(f"Chunk ID: {d.metadata['chunk_id']}")

print_docs(classic_docs, "CLASSIC RETRIEVAL (embedding similarity)")



CLASSIC RETRIEVAL (embedding similarity)

--- Result 1 ---
Section : Climate Delegated Act
Question: Will the technical screening criteria set out in the Climate Delegated Act be made stricter and updated over time?
Chunk ID: 0

--- Result 2 ---
Section : Climate Delegated Act
Question: How should GHG emissions for technical screening criteria be calculated (scope, methodologies etc.)?
Chunk ID: 5

--- Result 3 ---
Section : Climate Delegated Act
Question: How to deal with technical screening criteria that are not relevant to a specific activity mentioned in the description 
Chunk ID: 8

--- Result 4 ---
Section : Climate Delegated Act
Question: In relation to the technical screening criteria for DNSH to climate change adaptation, can the climate projections prepa
Chunk ID: 28


### Observations to think about

- **Classic** relies on the *semantic closeness* of the query to section+question+answer text.
- Dense retrieval can miss exact keyword matches that BM25 catches — this motivates the hybrid approach.


---
## 4. Generation – Prompt Engineering Comparison

We feed the same retrieved context into two prompt variants and inspect the answers.


In [17]:
from rag.query.chain import ask_question

q = "What should a company disclose if none of its economic activities are Taxonomy-eligible?"

basic = ask_question(q, prompt_type="basic", rag_type="classic")
improved = ask_question(q, prompt_type="intermediate", rag_type="classic")

print("BASIC PROMPT\n" + "-"*60)
print(basic["answer"])
print("\nINTERMEDIATE PROMPT\n" + "-"*60)
print(improved["answer"])


BASIC PROMPT
------------------------------------------------------------
If a company has no Taxonomy-eligible economic activities, it should disclose the information required in relation to its non-eligible economic activities. This includes filling in Section B of the templates provided in Annex II to the Disclosures Delegated Act, which relates to information on economic activities that are non-eligible.

INTERMEDIATE PROMPT
------------------------------------------------------------
If none of a company's economic activities are Taxonomy-eligible, the company should disclose the information required in relation to its non-eligible economic activities. Specifically, it can fill in Section B of the templates provided in Annex II to the Disclosures Delegated Act, which relates to information on economic activities that are non-eligible (as stated in the Taxonomy-Eligibility reporting, part 1).


**Improvements in the intermediate prompt:**
1. Explicit role framing ("specialized EU Taxonomy FAQ assistant").
2. Structured rule list including a refusal clause.
3. Instruction to cite specific sections.
4. Clear fallback wording so the model does not hallucinate.


---
## 5. Full Pipeline – Classic vs Hybrid

Compare answers and token usage implications across configurations.


In [18]:
questions = [
    "How does the EU Taxonomy define green economic activities?",
    "If a parent company already reports consolidated Taxonomy figures, can its NFRD subsidiary skip its own disclosures?",
    "Who won the 2018 FIFA World Cup?",  # unanswerable – tests refusal
]

for q in questions:
    print(f"\n{'='*70}")
    print(f"QUESTION: {q}")
    print('='*70)
    for rag_type in ("classic", "hybrid"):
        label = rag_type
        result = ask_question(q, prompt_type="intermediate", rag_type=rag_type, use_rerank=False)
        print(f"\n>> {label:<22} | sources: {[s['chunk_id'] for s in result['sources']]}")
        print(result["answer"][:300] + ("..." if len(result["answer"]) > 300 else ""))
    print()



QUESTION: How does the EU Taxonomy define green economic activities?

>> classic                | sources: [306, 310, 318, 321]
The EU Taxonomy defines green economic activities based on six EU environmental objectives: 

1. Climate change mitigation
2. Climate change adaptation
3. Sustainable use and protection of water and marine resources
4. Transition to a circular economy
5. Pollution prevention and control
6. Protectio...

>> hybrid                 | sources: [306, 310, 318, 308]
The EU Taxonomy defines green economic activities based on six EU environmental objectives: 

1. Climate change mitigation,
2. Climate change adaptation,
3. Sustainable use and protection of water and marine resources,
4. Transition to a circular economy,
5. Pollution prevention and control,
6. Prot...


QUESTION: If a parent company already reports consolidated Taxonomy figures, can its NFRD subsidiary skip its own disclosures?

>> classic                | sources: [208, 206, 257, 268]
Currently, under

---
## 6. Evaluation on the Golden Set (ragas)

We run the full evaluation suite on the 20-question test set.
This is the same logic exposed by `POST /evaluate` in the FastAPI app.

> ⏳ This can take a few minutes because every sample triggers multiple LLM and embedding calls (especially ragas metrics).


In [21]:
import asyncio
from rag.evaluate import evaluate_rag

# Run one configuration; change rag_type / prompt_type to explore others
result = asyncio.get_event_loop().run_until_complete(
    evaluate_rag(rag_type="classic", prompt_type="intermediate", use_rerank=False)
)

print(f"Configuration: {result['rag_type']} / {result['prompt_type']} / rerank={result['use_rerank']}")
print("\nAggregated Metrics:")
for metric, value in result["metrics"].items():
    print(f"  {metric:<22}: {value}")


RuntimeError: This event loop is already running

### What each metric means

| Metric | What it measures |
|---|---|
| **retrieval_hit** | Was the exact FAQ chunk we expected among the retrieved sources? (0–1) |
| **context_precision** | Of the retrieved chunks, how many were actually relevant? (ragas LLM judge) |
| **context_recall** | Did the retrieved chunks contain all information needed for the reference answer? |
| **faithfulness** | Is the generated answer fully grounded in the retrieved context? |
| **answer_relevancy** | Does the generated answer actually address the user's question? |
| **answer_rate** | Share of answerable questions the pipeline answered (not refused). Catches over-cautious refusals. |
| **refusal_rate** | Share of unanswerable questions correctly declined. |


---
### Baseline vs Hybrid Comparison

Run the evaluation on the **same golden set** for the three pipelines:
- **Baseline** = dense ChromaDB similarity only
- **Baseline+Rerank** = dense similarity + LLM-based rerank (costlier)
- **Hybrid** = BM25 + Dense + RRF + Cross-Encoder reranking

> ⏳ This takes ~10–15 minutes (20 questions × 3 methods).


In [ ]:
import asyncio
import pandas as pd
from rag.evaluate import evaluate_rag

# Pre-load the cross-encoder model in the main thread
# (prevents meta-tensor errors when asyncio threads first touch it)
from rag.query.methods.hybrid.cross_encoder_rerank import _ensure_loaded
_ensure_loaded()

configs = [
    {'name': 'Baseline',          'rag_type': 'classic', 'prompt_type': 'intermediate', 'use_rerank': False},
    {'name': 'Baseline+Rerank',   'rag_type': 'classic', 'prompt_type': 'intermediate', 'use_rerank': True},
    {'name': 'Hybrid',            'rag_type': 'hybrid',  'prompt_type': 'intermediate', 'use_rerank': False},
]

raw_results = []   # stores full sample objects
agg_rows = []      # stores aggregated metrics
for cfg in configs:
    print(f"\n⏳ Running {cfg['name']}...")
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    res = loop.run_until_complete(
        evaluate_rag(rag_type=cfg['rag_type'], prompt_type=cfg['prompt_type'], use_rerank=cfg['use_rerank'])
    )
    loop.close()
    row = {'Pipeline': cfg['name']}
    row.update(res['metrics'])

    agg_rows.append(row)
    raw_results.append({'name': cfg['name'], 'samples': res['samples']})
    print(f"   → retrieval_hit={row.get('retrieval_hit')}, faithfulness={row.get('faithfulness')}")



⏳ Running Baseline...


RuntimeError: Cannot run the event loop while another loop is running

#### Aggregated metrics per pipeline



In [ ]:
agg_df = pd.DataFrame(agg_rows)
agg_df = agg_df.set_index('Pipeline')
# Save to CSV so you can keep the numbers
agg_df.to_csv('01_comparison_aggregated.csv')
agg_df.T


#### How to read the comparison

- **retrieval_hit**: Hybrid should improve this if BM25 catches exact keyword matches that dense misses.
- **context_precision / context_recall**: Cross-encoder reranking should boost precision by pushing the most relevant docs to the top.
- **faithfulness**: Better retrieval → more grounded answers → higher faithfulness.
- **answer_relevancy**: Should stay stable or improve if the hybrid pipeline surfaces better context.
- **answer_rate / refusal_rate**: Check that hybrid does not hallucinate more (refusal should stay strong).


#### Per-question breakdown – all methods side-by-side

Each row is one question from the golden set.  \
`BL` = Baseline | `BL+RR` = Baseline+Rerank | `HY` = Hybrid  \
Hit=retrieval_hit | Prec=context_precision | Rec=context_recall | Faith=faithfulness | Rel=answer_relevancy


In [ ]:
per_q_rows = []
for i in range(len(raw_results[0]['samples'])):
    row = {
        'Q#': i + 1,
        'Question': raw_results[0]['samples'][i]['question'][:60] + ('…' if len(raw_results[0]['samples'][i]['question']) > 60 else ''),
        'Ans?': 'Y' if raw_results[0]['samples'][i]['answerable'] else 'N',
    }
    for r in raw_results:
        scores = r['samples'][i]['scores']
        short = {'Baseline': 'BL', 'Baseline+Rerank': 'BL+RR', 'Hybrid': 'HY'}[r['name']]
        row[f"{short}_Hit"]   = round(scores.get('retrieval_hit', 0),     2) if scores.get('retrieval_hit')   is not None else '-'
        row[f"{short}_Prec"]  = round(scores.get('context_precision', 0),2) if scores.get('context_precision') is not None else '-'
        row[f"{short}_Rec"]   = round(scores.get('context_recall', 0),   2) if scores.get('context_recall')   is not None else '-'
        row[f"{short}_Faith"] = round(scores.get('faithfulness', 0),     2) if scores.get('faithfulness')    is not None else '-'
        row[f"{short}_Rel"]   = round(scores.get('answer_relevancy', 0),2) if scores.get('answer_relevancy') is not None else '-'
    per_q_rows.append(row)

per_q_df = pd.DataFrame(per_q_rows)
per_q_df.to_csv('02_per_question_all_methods.csv', index=False)
per_q_df


### Inspect individual sample scores (Baseline)


In [ ]:
import pandas as pd

rows = []
for s in result["samples"]:
    row = {"question": s["question"], "answerable": s["answerable"], "answer": s["answer"][:120]}
    row.update({k: round(v, 3) if v is not None else None for k, v in s["scores"].items()})
    rows.append(row)

df = pd.DataFrame(rows)
df.head(20)
# Uncomment to focus on failures:
# df[df['retrieval_hit'] == 0] if 'retrieval_hit' in df.columns else df


---
## 7. Playground – Ask Your Own Question

Use the cell below to experiment with arbitrary questions and any combination of settings.


In [ ]:
# -------- CONFIGURE YOUR QUESTION HERE --------
my_question = "Can companies publish extra voluntary information alongside their mandatory Taxonomy-eligibility figures?"
my_rag_type = "hybrid"         # "classic" or "hybrid"
my_prompt_type = "intermediate"     # "basic" or "intermediate"
my_rerank = False                # True only works meaningfully with "classic"; "hybrid" always reranks via cross-encoder
# ----------------------------------------------

result = ask_question(
    my_question,
    prompt_type=my_prompt_type,
    rag_type=my_rag_type,
    use_rerank=my_rerank,
)

print(f"RAG type   : {my_rag_type}")
print(f"Prompt type: {my_prompt_type}")
print(f"Rerank     : {my_rerank}\n")
print("SOURCES:")
for s in result["sources"]:
    print(f"  [{s['chunk_id']}] {s['section']} — {s['question'][:80]}")
print("\nANSWER:")
print(result["answer"])


---
## 8. Hybrid Retrieval Pipeline

This section walks through the four-stage hybrid retrieval system step-by-step.

1. **BM25** – sparse lexical scoring (keyword matching).
2. **Dense** – embedding similarity over the vector store.
3. **Reciprocal Rank Fusion (RRF)** – fuses BM25 + dense rankings without calibration.
4. **Cross-Encoder Rerank** – scores every (query, doc) pair jointly with a transformer, then returns top 10.

You can also use `rag_type="hybrid"` directly in `ask_question()` (see Playground above).


In [ ]:
from rag.query.methods.hybrid.bm25 import bm25_retrieve_with_scores
from rag.query.methods.hybrid.dense import dense_retrieve_with_scores
from rag.query.methods.hybrid.reciprocal_rank_fusion import reciprocal_rank_fusion_with_scores
from rag.query.methods.hybrid.cross_encoder_rerank import cross_encoder_rerank_with_scores

QUESTION = "Are the technical screening criteria under the Climate Delegated Act going to be tightened and reviewed periodically?"

bm25_results = bm25_retrieve_with_scores(QUESTION, k=5)
dense_results = dense_retrieve_with_scores(QUESTION, k=5)
rrf_results = reciprocal_rank_fusion_with_scores(QUESTION, k=5)
ce_results = cross_encoder_rerank_with_scores(QUESTION, top_k=5)

def print_results(results, title, score_label):
    print(f"\n{title}")
    print("=" * 60)
    for rank, (doc, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.4f} {score_label}] {doc.metadata['question'][:80]}...")

print_results(bm25_results, "1) BM25 (top 5)", "BM25")
print_results(dense_results, "2) Dense (top 5)", "sim")
print_results(rrf_results, "3) RRF fused (top 5)", "RRF")
print_results(ce_results, "4) Cross-Encoder reranked (top 5)", "CE")


### Observations

- **BM25** excels at exact keyword overlap (e.g. "technical screening criteria").
- **Dense** surfaces semantically related chunks even with different wording.
- **RRF** boosts documents that appear in both lists without requiring score calibration.
- **Cross-Encoder** is slower but gives the most accurate relevance ordering because it sees query + document jointly.


---
## 9. Reflection Checklist

Use this section to verify your understanding of the system.

- [ ] **Parsing**: I can explain how the markdown is split into section → question → answer chunks.
- [ ] **Embeddings**: I know which model is used and where embeddings are stored.
- [ ] **Classic retrieval**: I can describe why top-k similarity search may retrieve semantically close but factually wrong chunks.
- [ ] **Hybrid retrieval**: I can explain how BM25 + dense + RRF + cross-encoder work together.
- [ ] **Prompt engineering**: I can list at least two differences between `basic` and `intermediate` prompts.
- [ ] **Reranking**: I know when it applies and the trade-off (extra LLM call vs better precision).
- [ ] **Evaluation**: I can interpret `retrieval_hit`, `faithfulness`, and `refusal_rate`.
- [ ] **Failure modes**: I can give an example of a question that the pipeline should refuse.


---
*End of notebook. Feel free to add more cells above to experiment with chunking strategies, alternative embedding models, or prompt variants.*
